# SHORT MAKER V3 - Story & Vocabulary

Tạo video Short kể chuyện hài hước/đời thường (khoảng 1 phút), dạy từ vựng.
- Ảnh dọc tràn viền, hiệu ứng zoom mượt mà.
- Phụ đề nền đen, chữ trắng, từ vựng được tô màu vàng nổi bật.

### Quy trình:
1. Chạy **Cell 1** (cài đặt, 1 lần duy nhất)
2. Nhập cấu hình ở **Cell 2**
3. Chạy **Cell 3** -> Video tự động tạo xong
4. Chạy **Cell 4** -> Tải video về máy

In [ ]:
# @title CELL 1: CÀI ĐẶT (chạy 1 lần, ~3 phút)
import os, subprocess, sys

print('Đang cài đặt thư viện...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'qwen-tts', 'huggingface_hub', 'pydub', 'openai-whisper',
    'pysrt', 'requests', 'playwright', 'nest_asyncio'], check=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

os.system('apt-get install -y -qq ffmpeg sox libsox-fmt-all 2>/dev/null')

from IPython.display import clear_output
clear_output()

import torch, soundfile, whisper, pysrt, requests, json, re, time, shutil, subprocess, random
from pathlib import Path
from qwen_tts import Qwen3TTSModel

print('TẤT CẢ THƯ VIỆN ĐÃ SẴN SÀNG!')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('   -> Upload file giọng mẫu (.wav) lên cột trái (nếu có)')
print('   -> Chạy Cell 2 để cấu hình')

In [ ]:
# @title CELL 2: CẤU HÌNH
# @markdown ---
# @markdown ### Chủ đề câu chuyện & từ vựng
topic = 'A relatable story about waking up late for work and rushing' # @param {type:'string'}
# @markdown ---
# @markdown ### API Keys
gemini_api_key = '' # @param {type:'string'}
# @markdown ---
# @markdown ### Kết nối TurboFlow (tạo ảnh)
bridge_url = '' # @param {type:'string'}
# @markdown ---
# @markdown ### Giọng đọc (Voice Clone)
file_giong_mau = 'yo.wav' # @param {type:'string'}
loi_thoai_giong_mau = 'Wait, have you ever noticed that time feels faster as we get older? It is kind of scary, right?' # @param {type:'string'}

assert topic.strip(), 'Nhập chủ đề!'
assert gemini_api_key.strip(), 'Nhập Gemini API key!'
assert bridge_url.strip(), 'Nhập Bridge URL!'

slug = re.sub(r'[^a-z0-9]+', '_', topic.lower()).strip('_')
print(f'Cấu hình OK! Project: {slug}')
print(f'   -> Chạy Cell 3 để tạo video')

In [ ]:
# @title CELL 3: TẠO VIDEO (tự động 100%)
import gc, base64, threading, math
from IPython.display import Audio, display, HTML
from pydub import AudioSegment

P = Path(f'/content/{slug}')
P.mkdir(parents=True, exist_ok=True)

print(f"{'='*60}")
print(f'STORY MAKER V3: {topic}')
print(f"{'='*60}")

print('\n[1/5] Sáng tạo cốt truyện & từ vựng (Gemini)...')
PROMPT = f"""You are a creative English teacher. Write a short, engaging, and funny story (approx 120-150 words) about: '{topic}'.
Seamlessly integrate 8 to 10 specific English vocabulary words or short phrases that the story aims to teach.
Provide:
1. story: The full story text.
2. target_words: Array of the exact 8-10 words/phrases to learn (must appear exactly as written in the story, case-insensitive).
3. image_prompts: Array of CONCISE image generation prompts (max 15 words per prompt). Number of prompts MUST exactly match 'target_words' plus 2 extra. To ensure a consistent style, every prompt MUST end with the EXACT phrase: ', 9:16 vertical, vibrant 2D flat illustration style, consistent character'.
Output ONLY valid JSON: {{"story": "...", "target_words": ["..."], "image_prompts": ["..."]}}
"""

body = {'contents': [{'role': 'user', 'parts': [{'text': PROMPT}]}], 'generationConfig': {'responseMimeType': 'application/json', 'temperature': 0.7}}
resp = requests.post(f'https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent?key={gemini_api_key.strip()}',
    headers={'Content-Type': 'application/json'}, json=body, timeout=30)
resp.raise_for_status()
raw = resp.json()['candidates'][0]['content']['parts'][0]['text'].strip()
raw = re.sub(r'^```json\\s*', '', raw)
raw = re.sub(r'\\s*```$', '', raw)
story_data = json.loads(raw)

story_text = story_data['story']
target_words = [w.lower() for w in story_data['target_words']]
image_prompts = story_data['image_prompts']
print(f'   Cốt truyện: {len(story_text.split())} từ.')
print(f'   Từ vựng học: {", ".join(target_words)}')
print(f'   Số lượng ảnh cần tạo: {len(image_prompts)}')

(P / 'story_data.json').write_text(json.dumps(story_data, indent=2, ensure_ascii=False), encoding='utf-8')

print('\n[2/5] Tạo hình ảnh (TurboFlow)...')
burl = bridge_url.strip().rstrip('/')
health = requests.get(f'{burl}/health', timeout=5)
assert health.ok, 'Bridge không hoạt động!'
server_time = health.json().get('time', time.time())

all_prompts = '\n\n'.join(image_prompts)
Path('/content/prompts_v3.txt').write_text(all_prompts, encoding='utf-8')
print('   Đã lưu prompts vào /content/prompts_v3.txt.')
print('   => Mở TurboFlow Extension, copy nội dung file này vào và bấm Start (chọn Aspect Ratio 9:16).')

since_ts = server_time - 10
downloaded_imgs = []
expected_imgs = len(image_prompts)
print(f'   Đang chờ {expected_imgs} ảnh từ TurboFlow...')

for poll in range(1200):
    time.sleep(2)
    try:
        imgs_resp = requests.get(f'{burl}/images', params={'since': since_ts}, timeout=10)
        if imgs_resp.ok:
            items = imgs_resp.json().get('items', [])
            new_items = [it for it in items if it['name'] not in [x['name'] for x in downloaded_imgs]]
            if new_items:
                for it in new_items:
                    img_path = P / f'img_{len(downloaded_imgs):02d}.jpg'
                    img_data = requests.get(f'{burl}/download', params={'name': it['name']}, timeout=30)
                    img_path.write_bytes(img_data.content)
                    downloaded_imgs.append({'name': it['name'], 'path': img_path})
                    print(f'\n      Đã tải: {img_path.name} ({len(downloaded_imgs)}/{expected_imgs})')
                if len(downloaded_imgs) >= expected_imgs:
                    print('   Đã tải đủ ảnh!')
                    break
            elif poll % 15 == 0:
                print(f'\r      Đang chờ ({len(downloaded_imgs)}/{expected_imgs})...', end='', flush=True)
    except: pass
print()
assert len(downloaded_imgs) >= expected_imgs, 'Không đủ ảnh!'

print('\n[3/5] Tạo giọng đọc (Qwen-TTS - Từng câu để ổn định giọng)...')
tts_model = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-Base', torch_dtype=torch.float16, device_map='cuda:0', attn_implementation='sdpa')
clone_prompt = tts_model.create_voice_clone_prompt(ref_audio=file_giong_mau, ref_text=loi_thoai_giong_mau.strip(), x_vector_only_mode=False)

audio_path = str(P / 'story_audio.wav')
# Chia nho cau chuyen thanh tung cau de tranh viec giong bi bien doi khi doc doan dai
sentences = re.split(r'(?<=[.!?]) +', story_text)
final_audio = AudioSegment.silent(duration=0)
for i, sent in enumerate(sentences):
    if not sent.strip(): continue
    print(f'   Đang đọc câu {i+1}/{len(sentences)}...')
    with torch.inference_mode():
        w_audio, sr = tts_model.generate_voice_clone(text=sent.strip(), voice_clone_prompt=clone_prompt)
    temp_wav = '/content/temp_sent.wav'
    soundfile.write(temp_wav, w_audio[0], sr)
    final_audio += AudioSegment.from_wav(temp_wav) + AudioSegment.silent(duration=300) # Nghi 300ms giua cac cau

final_audio.export(audio_path, format="wav")
del tts_model, clone_prompt; gc.collect(); torch.cuda.empty_cache()
print('   Đã tạo audio xong!')

print('\n[4/5] Trích xuất phụ đề (Whisper) & Tạo file ASS...')
wh_model = whisper.load_model('base', device='cuda:0')
result = wh_model.transcribe(audio_path, word_timestamps=True)
del wh_model; gc.collect(); torch.cuda.empty_cache()

ass_header = """[Script Info]
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920
WrapStyle: 1

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,60,&H00FFFFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,3,10,0,2,50,50,150,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
def srt_time(s):
    h = int(s // 3600); m = int((s % 3600) // 60); sec = int(s % 60); ms = int((s % 1) * 100)
    return f"{h}:{m:02d}:{sec:02d}.{ms:02d}"

ass_events = []
for segment in result['segments']:
    start_t = srt_time(segment['start'])
    end_t = srt_time(segment['end'])
    text = segment['text'].strip()
    for tw in target_words:
        pattern = re.compile(r'\b' + re.escape(tw) + r'\b', re.IGNORECASE)
        text = pattern.sub(lambda m: f"{{\\c&H00FFFF&\\b1}}{m.group(0)}{{\\c&HFFFFFF&\\b0}}", text)
    ass_events.append(f"Dialogue: 0,{start_t},{end_t},Default,,0,0,0,,{text}")

ass_path = P / 'subtitles.ass'
ass_path.write_text(ass_header + '\n'.join(ass_events), encoding='utf-8')
print('   Đã tạo xong phụ đề nổi bật từ vựng!')

print('\n[5/5] Dựng Video (Zoom Ảnh Nhẹ Ở Giữa & Ghép Nối)...')
audio_dur = len(AudioSegment.from_wav(audio_path)) / 1000.0
img_dur = audio_dur / len(downloaded_imgs)
list_txt = P / 'inputs.txt'
lines = []
for i, img in enumerate(downloaded_imgs):
    # Zoom rat nhe vao giua (zoom tu 1.0 toi max khoang 1.1)
    # x='iw/2-(iw/zoom)/2':y='ih/2-(ih/zoom)/2' dam bao luon giu o trung tam
    pan_filter = "zoompan=z='min(zoom+0.0005,1.1)':x='iw/2-(iw/zoom)/2':y='ih/2-(ih/zoom)/2':d={dur_frames}:s=1080x1920"
    dur_frames = int(img_dur * 30)
    out_vid = P / f'clip_{i:02d}.mp4'
    cmd = ['ffmpeg', '-y', '-loop', '1', '-i', str(img['path']), '-vf', pan_filter.format(dur_frames=dur_frames),
           '-c:v', 'libx264', '-t', str(img_dur), '-pix_fmt', 'yuv420p', '-r', '30', str(out_vid)]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    lines.append(f"file '{out_vid.name}'")
list_txt.write_text('\n'.join(lines))

merged_vid = P / 'merged.mp4'
subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(list_txt),
                '-c', 'copy', str(merged_vid)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

final_output = P / 'final_short.mp4'
cmd = ['ffmpeg', '-y', '-i', str(merged_vid), '-i', audio_path, 
       '-vf', f"ass='{ass_path}'",
       '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
       '-c:a', 'aac', '-b:a', '192k', '-shortest', str(final_output)]
subprocess.run(cmd, capture_output=True)
size_mb = final_output.stat().st_size / (1024*1024)
print(f'\n{"="*60}')
print(f'VIDEO HOÀN TẤT! ({size_mb:.1f} MB)')
print(f'   {final_output}')
print(f'{"="*60}')


In [ ]:
# @title CELL 4: TẢI VIDEO VỀ MÁY
from google.colab import files
files.download(str(final_output))
print('Đang tải video...')